In [ ]:
import pypsa
import pandas as pd
import numpy as np

### Region design

Region 1: Australia: High iron ore resources, expensive hydrogen and DRI, low steel demand
Region 2: Middle East: No iron ore resources, cheap hydrogen and DRI, low steel demand
Region 3: Europe: Some iron ore resources, expensive hydrogen and DRI, high steel demand

### Model

In [ ]:
# ====== CONFIGURATION PARAMETERS ======
# Regions
regions = ['1', '2', '3']

# Iron ore production costs (€/t)
ore_costs = [20, 30, 50]

# Iron ore production capacity per region (t)
ore_capacity = [200, 0, 0]

# Iron ore trade costs (€/t HBI/DRI)
iron_ore_trade_costs = [15, 15, 15]  # Regional variations in HBI transport costs

# Steel conversion costs by region (€/t steel)
steel_conversion_costs = {
    1: [300, 400, 500], 
    2: [200, 300, 400], 
    3: [500, 600, 700]
}
# Steel conversion limit (t per converter per region)
steel_conversion_limit = 50

# Steel conversion efficiency (tons steel per ton ore)
conversion_efficiency = 0.6

# Steel trade costs (€/t steel)
steel_trade_costs = [20, 20, 20]  # Uniform costs for all steel trade routes

# Steel demands per region (t)
steel_demands = [20, 10, 70]

# Trade capacity (large number for unlimited trade)
trade_capacity = 1000

# ====== MODEL SETUP ======
# Create a new PyPSA network
n = pypsa.Network()

# Add buses for iron ore (one per region)
n.madd("Bus", 
       names=[f"iron_ore_{region}" for region in regions],
       carrier="iron_ore")

# Add buses for steel (one per region)
n.madd("Bus", 
       names=[f"steel_{region}" for region in regions],
       carrier="steel")

# Add iron ore generators (one per region with different marginal costs)
n.madd("Generator",
       names=[f"iron_ore_gen_{region}" for region in regions],
       bus=[f"iron_ore_{region}" for region in regions],
       carrier="iron_ore_production",
       marginal_cost=ore_costs,
       p_nom=ore_capacity)

# Add links for iron ore trade between regions (with realistic HBI trade costs)
iron_ore_trade_pairs = [
    ("iron_ore_1", "iron_ore_2"),
    ("iron_ore_2", "iron_ore_3"),
    ("iron_ore_1", "iron_ore_3")
]

# Create bidirectional trade links
iron_ore_trade_names = []
iron_ore_trade_bus0 = []
iron_ore_trade_bus1 = []
iron_ore_trade_costs_list = []

for i, (bus0, bus1) in enumerate(iron_ore_trade_pairs):
    # Forward direction
    iron_ore_trade_names.append(f"iron_ore_trade_{bus0[-1]}_{bus1[-1]}")
    iron_ore_trade_bus0.append(bus0)
    iron_ore_trade_bus1.append(bus1)
    iron_ore_trade_costs_list.append(iron_ore_trade_costs[i])
    
    # Reverse direction
    iron_ore_trade_names.append(f"iron_ore_trade_{bus1[-1]}_{bus0[-1]}")
    iron_ore_trade_bus0.append(bus1)
    iron_ore_trade_bus1.append(bus0)
    iron_ore_trade_costs_list.append(iron_ore_trade_costs[i])

n.madd("Link",
       names=iron_ore_trade_names,
       bus0=iron_ore_trade_bus0,
       bus1=iron_ore_trade_bus1,
       marginal_cost=iron_ore_trade_costs_list,
       p_nom=trade_capacity,
       efficiency=1.0)

# Add links for steel trade between regions (with transport costs)
steel_trade_pairs = [
    ("steel_1", "steel_2"),
    ("steel_2", "steel_3"),
    ("steel_1", "steel_3")
]

# Create bidirectional trade links
steel_trade_names = []
steel_trade_bus0 = []
steel_trade_bus1 = []

for bus0, bus1 in steel_trade_pairs:
    # Forward direction
    steel_trade_names.append(f"steel_trade_{bus0[-1]}_{bus1[-1]}")
    steel_trade_bus0.append(bus0)
    steel_trade_bus1.append(bus1)
    
    # Reverse direction
    steel_trade_names.append(f"steel_trade_{bus1[-1]}_{bus0[-1]}")
    steel_trade_bus0.append(bus1)
    steel_trade_bus1.append(bus0)

n.madd("Link",
       names=steel_trade_names,
       bus0=steel_trade_bus0,
       bus1=steel_trade_bus1,
       marginal_cost=steel_trade_costs * 2,  # Duplicate for bidirectional links
       p_nom=trade_capacity,
       efficiency=1.0)

# Add conversion links from iron ore to steel (three per region with differentiated costs)
conversion_names = []
conversion_bus0 = []
conversion_bus1 = []
conversion_costs = []

for region_num, region in enumerate(regions, 1):
    for i, cost in enumerate(steel_conversion_costs[region_num]):
        conversion_names.append(f"ore_to_steel_{region}_{i+1}")
        conversion_bus0.append(f"iron_ore_{region}")
        conversion_bus1.append(f"steel_{region}")
        conversion_costs.append(cost)

n.madd("Link",
       names=conversion_names,
       bus0=conversion_bus0,
       bus1=conversion_bus1,
       marginal_cost=conversion_costs,
       efficiency=conversion_efficiency,
       p_nom=steel_conversion_limit)

# Add steel loads in each region
n.madd("Load",
       names=[f"steel_load_{region}" for region in regions],
       bus=[f"steel_{region}" for region in regions],
       p_set=steel_demands)

# Display network summary
print("=== NETWORK SUMMARY ===")
print(f"Buses: {len(n.buses)}")
print(f"Generators: {len(n.generators)}")
print(f"Links: {len(n.links)}")
print(f"Loads: {len(n.loads)}")

print("\n=== BUSES ===")
print(n.buses[['carrier']])

print("\n=== GENERATORS ===")
print(n.generators[['bus', 'carrier', 'marginal_cost', 'p_nom']])

print("\n=== LINKS ===")
print(n.links[['bus0', 'bus1', 'marginal_cost', 'efficiency', 'p_nom']])

print("\n=== LOADS ===")
print(n.loads[['bus', 'p_set']])

### Solving

In [ ]:
# Solve the optimization problem
print("\n=== SOLVING OPTIMIZATION ===")
n.optimize(solver_name='gurobi')

print(f"Objective value: {n.objective:.2f} €")

### Analysis

In [ ]:
# Display results
print("\n=== OPTIMIZATION RESULTS ===")

print("\nIron ore generation by region:")
for gen in n.generators.index:
    region = gen.split('_')[-1]
    generation = n.generators_t.p[gen].iloc[0] if not n.generators_t.p.empty else 0
    cost = n.generators.loc[gen, 'marginal_cost']
    print(f"Region {region}: {generation:.2f} t @ {cost} €/t")

print("\nIron ore trade flows:")
for link in n.links.index:
    if 'iron_ore_trade' in link:
        flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
        if abs(flow) > 0.01:  # Only show non-zero flows
            from_region = link.split('_')[3]
            to_region = link.split('_')[4]
            print(f"Region {from_region} → Region {to_region}: {flow:.2f} t")

print("\nSteel conversion by region:")
for link in n.links.index:
    if 'ore_to_steel' in link:
        flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
        if abs(flow) > 0.01:  # Only show non-zero flows
            region = link.split('_')[3]
            converter = link.split('_')[4]
            cost = n.links.loc[link, 'marginal_cost']
            steel_production = flow * n.links.loc[link, 'efficiency']
            print(f"Region {region}, Converter {converter}: {flow:.2f} t ore → {steel_production:.2f} t steel @ {cost} €/t steel")

print("\nSteel trade flows:")
for link in n.links.index:
    if 'steel_trade' in link:
        flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
        if abs(flow) > 0.01:  # Only show non-zero flows
            from_region = link.split('_')[2]
            to_region = link.split('_')[3]
            print(f"Region {from_region} → Region {to_region}: {flow:.2f} t")

print("\nSteel demand satisfaction:")
for load in n.loads.index:
    region = load.split('_')[-1]
    demand = n.loads.loc[load, 'p_set']
    print(f"Region {region}: {demand} t steel demand")

# Calculate total costs by component
print("\n=== COST BREAKDOWN ===")

# Iron ore production costs
ore_production_cost = 0
for gen in n.generators.index:
    generation = n.generators_t.p[gen].iloc[0] if not n.generators_t.p.empty else 0
    cost = n.generators.loc[gen, 'marginal_cost']
    ore_production_cost += generation * cost

# Steel conversion costs
steel_conversion_cost = 0
for link in n.links.index:
    if 'ore_to_steel' in link:
        flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
        steel_output = flow * n.links.loc[link, 'efficiency']
        cost = n.links.loc[link, 'marginal_cost']
        steel_conversion_cost += steel_output * cost

print(f"Iron ore production cost: {ore_production_cost:.2f} €")
print(f"Steel conversion cost: {steel_conversion_cost:.2f} €")
print(f"Total cost: {ore_production_cost + steel_conversion_cost:.2f} €")

### Additional analysis

In [ ]:
n.statistics.energy_balance(aggregate_bus=False)

### Plots

In [ ]:
# Iron Ore and Steel Production Flow Sankey Diagram
# This notebook creates a Sankey diagram from PyPSA optimization results

import pandas as pd
import plotly.graph_objects as go
import plotly.offline as pyo
from collections import defaultdict
import numpy as np

# Enable offline plotting in Jupyter
pyo.init_notebook_mode(connected=True)

def create_sankey_from_pypsa_results(n):
    """
    Create a Sankey diagram from PyPSA network results
    
    Parameters:
    n: PyPSA network object with solved optimization results
    """
    
    # Initialize data structures
    nodes = []
    node_dict = {}
    node_counter = 0
    
    # Flow data for Sankey
    sources = []
    targets = []
    values = []
    flow_labels = []
    
    def get_or_create_node(name, category=""):
        """Get existing node index or create new one"""
        nonlocal node_counter
        full_name = f"{name} ({category})" if category else name
        if full_name not in node_dict:
            node_dict[full_name] = node_counter
            nodes.append(full_name)
            node_counter += 1
        return node_dict[full_name]
    
    # 1. Process iron ore generation
    print("Processing iron ore generation...")
    for gen in n.generators.index:
        region = gen.split('_')[-1]
        generation = n.generators_t.p[gen].iloc[0] if not n.generators_t.p.empty else 0
        
        if generation > 0.01:  # Only include significant flows
            source_node = get_or_create_node(f"Iron Ore Source R{region}", "Generation")
            target_node = get_or_create_node(f"Iron Ore R{region}", "Supply")
            
            sources.append(source_node)
            targets.append(target_node)
            values.append(generation)
            cost = n.generators.loc[gen, 'marginal_cost']
            flow_labels.append(f"Generate: {generation:.1f}t @ {cost}€/t")
    
    # 2. Process iron ore trade flows
    print("Processing iron ore trade...")
    for link in n.links.index:
        if 'iron_ore_trade' in link:
            flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
            if abs(flow) > 0.01:
                from_region = link.split('_')[3]
                to_region = link.split('_')[4]
                
                source_node = get_or_create_node(f"Iron Ore R{from_region}", "Supply")
                target_node = get_or_create_node(f"Iron Ore R{to_region}", "Supply")
                
                sources.append(source_node)
                targets.append(target_node)
                values.append(abs(flow))
                flow_labels.append(f"Trade: {flow:.1f}t")
    
    # 3. Process steel conversion
    print("Processing steel conversion...")
    for link in n.links.index:
        if 'ore_to_steel' in link:
            ore_flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
            if abs(ore_flow) > 0.01:
                region = link.split('_')[3]
                converter = link.split('_')[4]
                efficiency = n.links.loc[link, 'efficiency']
                steel_production = ore_flow * efficiency
                cost = n.links.loc[link, 'marginal_cost']
                
                # Iron ore to steel conversion
                source_node = get_or_create_node(f"Iron Ore R{region}", "Supply")
                target_node = get_or_create_node(f"Steel R{region}", "Production")
                
                sources.append(source_node)
                targets.append(target_node)
                values.append(abs(ore_flow))
                flow_labels.append(f"Convert: {ore_flow:.1f}t ore → {steel_production:.1f}t steel @ {cost}€/t")
    
    # 4. Process steel trade flows
    print("Processing steel trade...")
    for link in n.links.index:
        if 'steel_trade' in link:
            flow = n.links_t.p0[link].iloc[0] if not n.links_t.p0.empty else 0
            if abs(flow) > 0.01:
                from_region = link.split('_')[2]
                to_region = link.split('_')[3]
                
                source_node = get_or_create_node(f"Steel R{from_region}", "Production")
                target_node = get_or_create_node(f"Steel R{to_region}", "Production")
                
                sources.append(source_node)
                targets.append(target_node)
                values.append(abs(flow))
                flow_labels.append(f"Trade: {flow:.1f}t")
    
    # 5. Process steel demand
    print("Processing steel demand...")
    for load in n.loads.index:
        region = load.split('_')[-1]
        demand = abs(n.loads.loc[load, 'p_set'])
        
        if demand > 0.01:
            source_node = get_or_create_node(f"Steel R{region}", "Production")
            target_node = get_or_create_node(f"Steel Demand R{region}", "Consumption")
            
            sources.append(source_node)
            targets.append(target_node)
            values.append(demand)
            flow_labels.append(f"Demand: {demand:.1f}t")
    
    # Create color scheme based on node categories
    node_colors = []
    for node in nodes:
        if "Generation" in node:
            node_colors.append("rgba(255, 127, 14, 0.8)")  # Orange for generation
        elif "Iron Ore" in node and "Supply" in node:
            node_colors.append("rgba(44, 160, 44, 0.8)")   # Green for iron ore supply
        elif "Steel" in node and "Production" in node:
            node_colors.append("rgba(31, 119, 180, 0.8)")  # Blue for steel production
        elif "Consumption" in node:
            node_colors.append("rgba(214, 39, 40, 0.8)")   # Red for consumption
        else:
            node_colors.append("rgba(148, 103, 189, 0.8)") # Purple for other
    
    # Create the Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            label=flow_labels,
            color="rgba(128, 128, 128, 0.4)"
        )
    )])
    
    # Update layout
    fig.update_layout(
        title_text="Iron Ore and Steel Production Flow Diagram",
        font_size=12,
        width=1200,
        height=800
    )
    
    return fig


# Create and display the Sankey diagram
try:
    fig = create_sankey_from_pypsa_results(n)
    fig.show()
    
    # Optional: Save as HTML file
    # fig.write_html("iron_steel_sankey.html")
    # print("Sankey diagram saved as 'iron_steel_sankey.html'")
    
except NameError:
    print("Please ensure your PyPSA network object is named 'n' and contains solved results")
    print("The network should have generators, links, and loads with the naming convention shown in your output")


print("Notebook ready! Run create_sankey_from_pypsa_results(n) with your PyPSA network object.")